In [1]:
import json
import re
import csv
from pathlib import Path

In [2]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

In [4]:
DATASET_FOLDER = Path("/Users/megan/Downloads/PMC-Patients-full")

In [5]:
OUTPUT_FOLDER = DATASET_FOLDER / "tfidf_output2"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [6]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /Users/megan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/megan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/megan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [7]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [8]:
def preprocess_text(text):
    # convert all text to lowercase so "Patients" is the same as "patient"
    text = text.lower()

    # remove characters that are not lowercase, numbers, or normal spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    # tokenize terms
    words = text.split()

    # remove stop words (e.g., the, or, with) and lemmatization (patients and patient is same term)
    processed_words = []

    for word in words:
        if word not in stop_words:
            lemma = lemmatizer.lemmatize(word)
            processed_words.append(lemma)

    return " ".join(processed_words)

In [9]:
documents = []
document_ids = []

In [10]:
json_files = sorted(DATASET_FOLDER.rglob("*.json"))

In [11]:
print(f"Found {len(json_files)} JSON files.")

Found 210071 JSON files.


In [12]:
# I used the WordNet lemmatizer and now will extract the JSON documents
for file_path in json_files:

    if OUTPUT_FOLDER in file_path.parents:
        continue

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        records = data if isinstance(data, list) else [data]

        for record in records:
            if not isinstance(record, dict):
                continue

            title = record.get("title", "")
            patient_text = record.get("patient", "")

            if not isinstance(title, str):
                title = ""

            if not isinstance(patient_text, str):
                patient_text = ""

            original_text = f"{title} {patient_text}".strip()

            
            if not original_text:
                continue

            processed_text = preprocess_text(original_text)

            documents.append(processed_text)

            # save identifiers for later
            document_ids.append({
                "filename": str(file_path.relative_to(DATASET_FOLDER)),
                "patient_id": record.get("patient_id", ""),
                "PMID": record.get("PMID", "")
            })

    except (json.JSONDecodeError, OSError) as error:
        print(f"Could not read {file_path}: {error}")


print(f"Successfully extracted {len(documents)} documents.")

Successfully extracted 210069 documents.


In [13]:
if not documents:
    raise ValueError(
        "No usable documents were found. "
        "Check the dataset folder and JSON structure."
    )

In [14]:
vectorizer = TfidfVectorizer(
    smooth_idf=True,
    sublinear_tf=True,
    norm="l2"
)

In [15]:
tfidf_matrix = vectorizer.fit_transform(documents)

print("TF-IDF matrix constructed successfully.")
print("Matrix dimensions:", tfidf_matrix.shape)

TF-IDF matrix constructed successfully.
Matrix dimensions: (210069, 286964)


In [16]:
save_npz(
    OUTPUT_FOLDER / "tfidf_matrix2.npz",
    tfidf_matrix
)

In [17]:
vocabulary = vectorizer.get_feature_names_out()

In [18]:
with open(
    OUTPUT_FOLDER / "tfidf_vocabulary2.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(vocabulary.tolist(), file, ensure_ascii=False)

In [19]:
with open(
    OUTPUT_FOLDER / "document_metadata2.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=["filename", "patient_id", "PMID"]
    )

    writer.writeheader()
    writer.writerows(document_ids)


In [20]:
idf_values = vectorizer.idf_.tolist()

In [21]:
with open(
    OUTPUT_FOLDER / "idf_values2.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        dict(zip(vocabulary.tolist(), idf_values)),
        file,
        ensure_ascii=False
    )

In [22]:
print(f"All output files saved to: {OUTPUT_FOLDER}")

All output files saved to: /Users/megan/Downloads/PMC-Patients-full/tfidf_output2


In [31]:
# now I want to create a smaller matrix because the complete one is a very large file and change to csv so I can view w/o python
import numpy as np
import pandas as pd
from pathlib import Path
import scipy.sparse as sp

In [33]:
tfidf_matrix = sp.load_npz('/Users/megan/Downloads/PMC-Patients-full/tfidf_output2/tfidf_matrix2.npz')
small_matrix = tfidf_matrix[:1000, :1000]

small_features = vectorizer.get_feature_names_out()[:1000]

small_df = pd.DataFrame(
    small_matrix.toarray(),
    columns=small_features
)

small_df.to_csv(
    OUTPUT_FOLDER / "tfidf_matrix_small2.csv",
    index=False
)